1.- Carga inicial de datos

In [3]:
import numpy as np
import pandas as pd

df_customers_all = pd.read_csv('../../documents/datasets/customers_dataset.csv') # Correcto
df_order_items_all = pd.read_csv('../../documents/datasets/order_items_dataset.csv') # Correcto
df_order_payments_all = pd.read_csv('../../documents/datasets/order_payments_dataset.csv') # Correcto
df_order_reviews_all = pd.read_csv('../../documents/datasets/order_reviews_dataset.csv') # Correcto
df_orders_all = pd.read_csv('../../documents/datasets/orders_dataset.csv') # Correcto
df_product_category_name_translation_all = pd.read_csv('../../documents/datasets/product_category_name_translation.csv') # Correcto
df_products_all = pd.read_csv('../../documents/datasets/products_dataset.csv') # Correcto
df_sellers_all = pd.read_csv('../../documents/datasets/sellers_dataset.csv') # Correcto

2.- Número de clientes por ciudad

In [4]:
    # Cambiamos el tipo de order_pruchase_timestamp para poder operar con él y decidir el rango de fechas dinámico desde el dashboard de Streamlit
df_orders_all["order_purchase_timestamp"] = pd.to_datetime(df_orders_all["order_purchase_timestamp"])

    # Definimos fecha de inicio y de fin
st_date = pd.to_datetime(df_orders_all["order_purchase_timestamp"].min())
ed_date = pd.to_datetime(df_orders_all["order_purchase_timestamp"].max())

    # Filtramos los pedidos por las fechas
df_filtered = df_orders_all[(df_orders_all["order_purchase_timestamp"] >= st_date) & (df_orders_all["order_purchase_timestamp"] <= ed_date)]

    # Hacemos un join con el DataFrame de clientes por el 'customer_id', para poder acceder a las ciudades y a los estados
df_filtered_orders = pd.merge(df_customers_all, df_filtered, on="customer_id")

# Normalizamos el nombre de la ciudad
df_filtered_orders['customer_city'] = df_filtered_orders['customer_city'].str.capitalize()

# Normalizamos los datos y realizamos el filtrado para quedarnos con los datos que nos interesan mostrar, agrupamos por el estado y ciudad y obtenemos
 # los clientes unicos, ordenamos de manera descendente, reseteamos índices para poder acceder a los datos más fácilmente y renombramos las columnas que necesitamos
df_filtered_orders = df_filtered_orders.groupby(["customer_state", 
                                                     "customer_city"])['customer_unique_id'].nunique().sort_values(ascending=False).reset_index(
                                                         name="Nº Clientes por ciudad").rename(
                                                             columns={"customer_state": "Estado", 
                                                                      "customer_city": "Ciudad"})

# return df_filtered_orders.head(n=number) # Devolvemos el DataFrame según el número que se nos pase en streamlit
df_filtered_orders

,Estado,Ciudad,Nº Clientes por ciudad
0,SP,Sao paulo,14984
1,RJ,Rio de janeiro,6620
2,MG,Belo horizonte,2672
3,DF,Brasilia,2069
4,PR,Curitiba,1465
...,...,...,...
4305,SP,Sao sebastiao da serra,1
4306,SP,Sarutaia,1
4307,AL,Marechal deodoro,1
4308,AL,Maribondo,1


3.- Número de pedidos por ciudad

In [6]:


    # Cambiamos el tipo de order_pruchase_timestamp para poder operar con él y decidir el rango de fechas dinámico desde el dashboard de Streamlit
df_orders_all[ "order_purchase_timestamp"] = pd.to_datetime(df_orders_all["order_purchase_timestamp"])

    # Definimos fecha de inicio y de fin
st_date = pd.to_datetime(df_orders_all["order_purchase_timestamp"].min())
ed_date = pd.to_datetime(df_orders_all["order_purchase_timestamp"].max())

     # Filtramos los pedidos por las fechas
df_filtered = df_orders_all[(df_orders_all["order_purchase_timestamp"] >= st_date) & (df_orders_all["order_purchase_timestamp"] <= ed_date)]
    
    # Declaramos el total de pedidos 
total_pedidos = df_filtered["order_id"].count()

    # Hacemos un join con el DataFrame de clientes por el 'customer_id', para poder acceder a las ciudades y a los estados
df_filtered_orders = pd.merge(df_customers_all, df_filtered, on="customer_id")

    # Normalizamos el nombre de la ciudad
df_filtered_orders['customer_city'] = df_filtered_orders['customer_city'].str.capitalize()

    # Normalizamos los datos y realizamos el filtrado para quedarnos con los datos que nos interesan mostrar, agrupamos por el estado y ciudad y realizamos una
    # función de agregación donde obtenemos los clientes unicos y la cantidad de pedidos total de la ciudad, reseteamos índices para poder acceder a los datos más fácilmente 
    # y renombramos las columnas que necesitamos
df_filtered_orders = df_filtered_orders.groupby(["customer_state", 
                                                     "customer_city"]).agg(**{"Nº Clientes por ciudad": ("customer_unique_id", "nunique"),
                                                                               "Nº Pedidos por ciudad": ("order_id", "count")}).reset_index().rename(
                                                                                       columns={"customer_state" : "Estado",
                                                                                                 "customer_city" : "Ciudad"})

    # Definimos el porcentaje de los pedidos obtenidos respecto al total de pedidos que hemos definido anteriormente
df_filtered_orders["% Pedidos respecto al total"] = round(df_filtered_orders["Nº Pedidos por ciudad"] / total_pedidos * 100, 2)

    # Creamos una copia del dataframe anterior para poder devolverlo más adelante, en este dataframe calcularemos el ratio de pedidos por cliente
df_customer_orders = df_filtered_orders.copy()

    # Calculamos el ratio diviendo los pedidos de la ciudad entre los clientes de la ciudad
df_customer_orders['Ratio de pedidos por cliente'] = (df_filtered_orders["Nº Pedidos por ciudad"] / df_filtered_orders["Nº Clientes por ciudad"])
    
df_customer_orders
                      

,Estado,Ciudad,Nº Clientes por ciudad,Nº Pedidos por ciudad,% Pedidos respecto al total,Ratio de pedidos por cliente
0,AC,Brasileia,1,1,0.00,1.0
1,AC,Cruzeiro do sul,3,3,0.00,1.0
2,AC,Epitaciolandia,1,1,0.00,1.0
3,AC,Manoel urbano,1,1,0.00,1.0
4,AC,Porto acre,1,1,0.00,1.0
...,...,...,...,...,...,...
4305,TO,Silvanopolis,1,1,0.00,1.0
4306,TO,Sitio novo do tocantins,1,2,0.00,2.0
4307,TO,Taguatinga,3,3,0.00,1.0
4308,TO,Tocantinopolis,7,7,0.01,1.0
